In [4]:
import sys
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.abspath('..'))
from src.data_loader import TimeSeriesDataLoader

loader = TimeSeriesDataLoader()
df = loader.load_and_clean('../data/raw/Internal Submission Hourly 2024.csv')

col_ci = 'Carbon intensity gCO₂eq/kWh (direct)'
col_cfe = 'Carbon-free energy percentage (CFE%)'
col_re = 'Renewable energy percentage (RE%)'

print("Data ready for EDA! Shape:", df.shape)

Loading data from ../data/raw/Internal Submission Hourly 2024.csv...
Data cleaned. Total baris: 8784
Data ready for EDA! Shape: (8784, 3)


In [5]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                    subplot_titles=("Hourly Carbon Intensity (gCO₂eq/kWh)", 
                                    "Carbon-Free Energy (%)", 
                                    "Renewable Energy (%)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=df.index, y=df[col_ci], name="Carbon Intensity", line=dict(color='firebrick', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df[col_cfe], name="CFE %", line=dict(color='royalblue', width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df[col_re], name="RE %", line=dict(color='mediumseagreen', width=1)), row=3, col=1)

fig.update_layout(height=800, title_text="Overview Dinamika Energi PLN 2024", showlegend=False, hovermode="x unified")
fig.show()

In [7]:
df_time = df.copy()
df_time['Hour'] = df_time.index.hour
df_time['DayOfWeek'] = df_time.index.day_name()

# Sort days
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_time['DayOfWeek'] = pd.Categorical(df_time['DayOfWeek'], categories=days_order, ordered=True)

# Agregasi Harian (Jam 00 - 23)
hourly_avg = df_time.groupby('Hour')[col_ci].mean().reset_index()

# Agregasi Mingguan (Senin - Minggu)
weekly_avg = df_time.groupby('DayOfWeek')[col_ci].mean().reset_index()

# Visualisasi
fig_season = make_subplots(rows=1, cols=2, subplot_titles=("Rata-rata Emisi per Jam (Daily Seasonality)", "Rata-rata Emisi per Hari (Weekly Seasonality)"))

fig_season.add_trace(go.Scatter(x=hourly_avg['Hour'], y=hourly_avg[col_ci], mode='lines+markers', line=dict(color='orange')), row=1, col=1)
fig_season.add_trace(go.Bar(x=weekly_avg['DayOfWeek'], y=weekly_avg[col_ci], marker_color='indianred'), row=1, col=2)

fig_season.update_layout(height=400, title_text="Analisis Multiple Seasonality Carbon Intensity", showlegend=False)
fig_season.update_xaxes(title_text="Jam", tickmode='linear', dtick=2, row=1, col=1)
fig_season.update_xaxes(title_text="Hari", row=1, col=2)
fig_season.show()

In [12]:
# Hitung matriks korelasi
corr_matrix = df[[col_ci, col_cfe, col_re]].corr()

# Heatmap
fig_corr = px.imshow(corr_matrix, 
                     text_auto='.3f', 
                     aspect="auto", 
                     color_continuous_scale='RdBu_r', # Red to Blue (merah korelasi positif, biru negatif)
                     title="Heatmap Korelasi: Variabel Multivariate")

fig_corr.update_layout(height=800, width=1200)
fig_corr.show()